# Dead Letter / Escalation Pattern | Agent Safety & Resilience

In [1]:
# Dead Letter Queue: API Ingestion with Retry, DLQ, and Human Escalation
# Lifecycle: succeed | retry-then-succeed | exhaust retries -> DLQ -> escalation alert
from dataclasses import dataclass, field
from typing import List, Callable
from datetime import datetime
import time

In [2]:
@dataclass
class DeadLetter:
    task_id: str
    endpoint: str
    error: str
    attempts: int
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

class DeadLetterQueue:
    def __init__(self, max_retries: int = 3, escalation_threshold: int = 2):
        self.max_retries = max_retries
        self.escalation_threshold = escalation_threshold
        self.dlq: List[DeadLetter] = []
        self.successes: List[str] = []

    def execute_with_retry(self, task_id: str, endpoint: str, func: Callable) -> str:
        """Execute with exponential backoff. On exhaustion, send to DLQ."""
        for attempt in range(1, self.max_retries + 1):
            try:
                result = func()
                self.successes.append(task_id)
                status = f"(after {attempt} attempt(s))" if attempt > 1 else "(first try)"
                print(f"  [{task_id}] SUCCESS {status}: {result}")
                return result
            except Exception as exc:
                last_error = str(exc)
                wait = 0.01 * (2 ** (attempt - 1))  # Exponential backoff: 0.01, 0.02, 0.04s
                print(f"  [{task_id}] Attempt {attempt}/{self.max_retries} FAILED: {exc} | backoff {wait:.2f}s")
                if attempt < self.max_retries:
                    time.sleep(wait)
        # All retries exhausted -> dead letter queue
        dl = DeadLetter(task_id, endpoint, last_error, self.max_retries)
        self.dlq.append(dl)
        print(f"  [{task_id}] -> DEAD LETTER QUEUE after {self.max_retries} attempts")
        self._check_escalation()
        return f"DLQ:{task_id}"

    def _check_escalation(self):
        """Escalate to human when DLQ hits threshold."""
        if len(self.dlq) >= self.escalation_threshold:
            print(f"\n{'='*60}")
            print(f"HUMAN ESCALATION ALERT: {len(self.dlq)} tasks in dead letter queue!")
            for dl in self.dlq:
                print(f"  - {dl.task_id} | {dl.endpoint} | {dl.error} | {dl.timestamp}")
            print(f"{'='*60}\n")

In [3]:
# --- Simulate an unreliable API data ingestion pipeline ---
pipeline = DeadLetterQueue(max_retries=3, escalation_threshold=2)

def make_api_fetcher(endpoint: str, fail_count: int):
    """Fetcher that fails `fail_count` times then succeeds."""
    state = {"calls": 0}
    def fetch():
        state["calls"] += 1
        if state["calls"] <= fail_count: raise ConnectionError(f"503 from {endpoint}")
        return f"{{'data': 'rows from {endpoint}'}}"
    return fetch

print("--- Task 1: Reliable endpoint ---")
pipeline.execute_with_retry("ingest-001", "/api/users", make_api_fetcher("/api/users", 0))
print("\n--- Task 2: Flaky endpoint (recovers after 2 failures) ---")
pipeline.execute_with_retry("ingest-002", "/api/orders", make_api_fetcher("/api/orders", 2))
print("\n--- Task 3: Down endpoint (permanent failure -> DLQ) ---")
pipeline.execute_with_retry("ingest-003", "/api/payments", make_api_fetcher("/api/payments", 10))
print("\n--- Task 4: Another failure -> DLQ threshold triggers escalation ---")
pipeline.execute_with_retry("ingest-004", "/api/inventory", make_api_fetcher("/api/inventory", 10))
print(f"\nSummary: {len(pipeline.successes)} succeeded, {len(pipeline.dlq)} in DLQ")

--- Task 1: Reliable endpoint ---
  [ingest-001] SUCCESS (first try): {'data': 'rows from /api/users'}

--- Task 2: Flaky endpoint (recovers after 2 failures) ---
  [ingest-002] Attempt 1/3 FAILED: 503 from /api/orders | backoff 0.01s
  [ingest-002] Attempt 2/3 FAILED: 503 from /api/orders | backoff 0.02s
  [ingest-002] SUCCESS (after 3 attempt(s)): {'data': 'rows from /api/orders'}

--- Task 3: Down endpoint (permanent failure -> DLQ) ---
  [ingest-003] Attempt 1/3 FAILED: 503 from /api/payments | backoff 0.01s
  [ingest-003] Attempt 2/3 FAILED: 503 from /api/payments | backoff 0.02s
  [ingest-003] Attempt 3/3 FAILED: 503 from /api/payments | backoff 0.04s
  [ingest-003] -> DEAD LETTER QUEUE after 3 attempts

--- Task 4: Another failure -> DLQ threshold triggers escalation ---
  [ingest-004] Attempt 1/3 FAILED: 503 from /api/inventory | backoff 0.01s
  [ingest-004] Attempt 2/3 FAILED: 503 from /api/inventory | backoff 0.02s
  [ingest-004] Attempt 3/3 FAILED: 503 from /api/inventory | 